# DustyCult July 1 Dippers

Run DustyCult quick fits for the July 1 review candidates labeled `dipper` in `output/runs/dat3-full-extended_2026-07-01-v4/review/review.db`.

This notebook writes fit metadata and posterior predictive curves back into the review DB tables used by the review app:

- `dustycult_fits`
- `dustycult_predictive_curves`
- `output/runs/dat3-full-extended_2026-07-01-v4/review/dustycult/<candidate>/quick/`


## Setup

Run these first. The fitting cells near the end are the only cells that intentionally update the review DB and DustyCult artifact directories.

In [1]:
from __future__ import annotations

import json
import math
import sqlite3
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

root_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        p for p in root_candidates
        if (p / 'pyproject.toml').exists()
        and (p / 'malca' / 'review' / 'dustycult.py').exists()
    ),
    Path.cwd(),
)

for path in (REPO_ROOT, REPO_ROOT / 'malca'):
    text = str(path.resolve())
    if text not in sys.path:
        sys.path.insert(0, text)

from malca.io.notebook_paths import resolve_local_lightcurve_path
from malca.review.dustycult import (
    check_dustycult_available,
    control_defaults_for_candidate,
    load_dustycult_curve,
    load_dustycult_fits,
    run_dustycult_fit,
)
from malca.review.dustycult_display import (
    build_dustycult_fit_figure,
    dustycult_fit_metadata_rows,
    dustycult_geometry_rows,
    dustycult_posterior_rows,
    select_dustycult_display_row,
)
from malca.review.dustycult_visualization import build_dustycult_occulter_figure
from malca.review.store import db_connect

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 160)
pd.set_option('display.width', 240)


/opt/homebrew/Caskroom/miniconda/base/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration And Preflight

`MAX_CANDIDATES = 1` is the smoke-test default. The full-run cell below resets it to `None` and skips existing `ok` or `warning` quick fits.

In [2]:
RUN_DIR = REPO_ROOT / 'output' / 'runs' / 'dat3-full-extended_2026-07-01-v4'
DB_PATH = RUN_DIR / 'review' / 'review.db'
LIGHTCURVE_DIR = RUN_DIR / 'bundle_assets' / 'lightcurves'
PLOT_DIR = RUN_DIR / 'plots'
RESULTS_DIR = RUN_DIR / 'results'

MODES = ('quick',)
GOOD_STATUSES = {'ok', 'warning'}
RERUN_EXISTING = False
MAX_CANDIDATES = 1
EXPECTED_DIPPERS = 71
EXPECTED_MISSING_QUICK = 69
JULIA = 'julia'
DUSTYCULT_PROJECT = REPO_ROOT / 'external' / 'dustycult'
EXAMPLE_PLOT_MODE = 'quick'

RUN_PARAMS_PATH = RUN_DIR / 'run_params.json'
RUN_PARAMS = json.loads(RUN_PARAMS_PATH.read_text()) if RUN_PARAMS_PATH.exists() else {}

availability = check_dustycult_available(project_path=DUSTYCULT_PROJECT, julia=JULIA)

print(f'REPO_ROOT         = {REPO_ROOT}')
print(f'RUN_DIR           = {RUN_DIR}')
print(f'DB_PATH           = {DB_PATH}')
print(f'LIGHTCURVE_DIR    = {LIGHTCURVE_DIR}')
print(f'PLOT_DIR          = {PLOT_DIR}')
print(f'DUSTYCULT_PROJECT = {DUSTYCULT_PROJECT}')
print(f'DB exists         = {DB_PATH.exists()}')
print(f'LC dir exists     = {LIGHTCURVE_DIR.is_dir()}')
print(f'run_params exists = {RUN_PARAMS_PATH.exists()}')
print(f'Julia executable  = {availability.julia}')
print(f'DustyCult status  = {availability.message}')

if not DB_PATH.exists():
    raise FileNotFoundError(DB_PATH)
if not LIGHTCURVE_DIR.is_dir():
    raise FileNotFoundError(LIGHTCURVE_DIR)
if not availability.ok:
    raise RuntimeError(availability.message)


REPO_ROOT         = /Users/calder/code/malca
RUN_DIR           = /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4
DB_PATH           = /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/review/review.db
LIGHTCURVE_DIR    = /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/bundle_assets/lightcurves
PLOT_DIR          = /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/plots
DUSTYCULT_PROJECT = /Users/calder/code/malca/external/dustycult
DB exists         = True
LC dir exists     = True
run_params exists = True
Julia executable  = julia
DustyCult status  = DustyCult is available


## Load July 1 Dipper Candidates

This uses the strict selector `reviews.event_class = 'dipper'`.

In [3]:
DIPPER_WHERE = "r.event_class = 'dipper'"

with db_connect(DB_PATH) as conn:
    reviewed_dippers = pd.read_sql_query(
        f'''
        SELECT
            c.*,
            r.event_class,
            r.workflow_status,
            r.disposition,
            r.morphology_primary,
            r.morphology_secondary,
            r.morphology_secondary_json,
            r.physical_primary,
            r.physical_secondary,
            r.classification_confidence,
            r.priority_tags_json,
            r.notes AS review_notes,
            r.updated_at AS review_updated_at
        FROM reviews r
        JOIN candidates c USING(candidate_id)
        WHERE {DIPPER_WHERE}
        ORDER BY r.updated_at DESC, r.candidate_id
        ''',
        conn,
    )
    existing_fits = pd.read_sql_query(
        f'''
        SELECT candidate_id, mode, status, updated_at, runtime_sec, t0_jd, start_jd, end_jd,
               n_input_points, n_curve_points, artifact_dir, error
        FROM dustycult_fits
        WHERE candidate_id IN (
            SELECT r.candidate_id FROM reviews r WHERE {DIPPER_WHERE}
        )
        ORDER BY candidate_id, mode
        ''',
        conn,
    )

quick_fits = existing_fits[existing_fits['mode'].astype(str).eq('quick')].copy() if not existing_fits.empty else pd.DataFrame()
good_quick_ids = set(
    quick_fits.loc[quick_fits['status'].astype(str).isin(GOOD_STATUSES), 'candidate_id'].astype(str)
) if not quick_fits.empty else set()
missing_quick_count = int(len(reviewed_dippers) - len(good_quick_ids))

print(f'July 1 dippers: {len(reviewed_dippers)}')
print(f'Existing quick fits with ok/warning status: {len(good_quick_ids)}')
print(f'Missing quick fits by skip policy: {missing_quick_count}')

if len(reviewed_dippers) != EXPECTED_DIPPERS:
    display(Markdown(f'**Warning:** expected `{EXPECTED_DIPPERS}` dippers, found `{len(reviewed_dippers)}`.'))
if missing_quick_count != EXPECTED_MISSING_QUICK:
    display(Markdown(f'**Warning:** expected `{EXPECTED_MISSING_QUICK}` missing quick fits, found `{missing_quick_count}`.'))

display_cols = [
    'candidate_id', 'event_class', 'workflow_status', 'disposition',
    'morphology_primary', 'morphology_secondary', 'physical_primary',
    'dipper_score', 'dipper_n_valid_dips', 'dip_run_count', 'dip_best_morph',
]
display(reviewed_dippers[[col for col in display_cols if col in reviewed_dippers.columns]])
display(existing_fits if not existing_fits.empty else Markdown('No existing DustyCult fits for these dippers.'))


July 1 dippers: 71
Existing quick fits with ok/warning status: 2
Missing quick fits by skip policy: 69


,candidate_id,event_class,workflow_status,disposition,morphology_primary,morphology_secondary,physical_primary,dipper_score,dipper_n_valid_dips,dip_run_count,dip_best_morph
0,stv_360777826205,dipper,reviewed,keep,dimming_event,big_dipper,None,21.400308,67.0,4.0,noise
1,stv_197569226514,dipper,reviewed,keep,dimming_event,asymmetric_dip,None,-1.597394,11.0,6.0,noise
2,stv_395138016907,dipper,reviewed,keep,dimming_event,single_dip,None,20.315220,58.0,15.0,gaussian
3,stv_163209415214,dipper,reviewed,keep,dimming_event,sharp_dip,None,18.589905,20.0,6.0,skew_gaussian
4,stv_403727411589,dipper,reviewed,keep,dimming_event,single_dip,None,18.884118,37.0,4.0,gaussian
5,stv_25770316308,dipper,reviewed,keep,dimming_event,quasi_periodic_dips,None,20.531731,23.0,1.0,gaussian
6,stv_584116028406,dipper,reviewed,keep,dimming_event,single_dip,None,-0.523791,9.0,4.0,gaussian
7,stv_154620038181,dipper,reviewed,keep,dimming_event,sharp_dip,None,18.878795,28.0,12.0,gaussian
8,stv_180388640882,dipper,reviewed,keep,dimming_event,sharp_dip,None,21.756072,40.0,3.0,gaussian
9,stv_17181027305,dipper,reviewed,keep,dimming_event,single_dip,None,-3.302735,4.0,1.0,gaussian


,candidate_id,mode,status,updated_at,runtime_sec,t0_jd,start_jd,end_jd,n_input_points,n_curve_points,artifact_dir,error
0,stv_369367304600,quick,ok,2026-07-09T05:52:31.639486+00:00,16.276035,2.460712e+06,2.460704e+06,2.460725e+06,4,4,/Users/calder/code/malca/output/runs/dat3-full...,
1,stv_566936725849,quick,warning,2026-07-09T07:36:59.650249+00:00,2304.978950,2.460070e+06,2.460003e+06,2.460137e+06,76,76,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...


## Helpers

These mirror the review app path resolution and DustyCult fit calls, but keep execution serial for DB safety.

In [4]:
def _finite_float(value, default=None):
    try:
        if value is None or pd.isna(value):
            return default
    except Exception:
        if value is None:
            return default
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default
    return number if math.isfinite(number) else default


def clean_value(value):
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def row_payload(row: pd.Series) -> dict[str, object]:
    payload: dict[str, object] = {}
    raw = row.get('payload_json')
    if isinstance(raw, str) and raw.strip():
        try:
            payload.update(json.loads(raw))
        except Exception:
            pass
    for key, value in row.items():
        cleaned = clean_value(value)
        if cleaned is not None:
            payload[key] = cleaned
    payload['candidate_id'] = str(row['candidate_id'])
    return payload


def resolve_candidate_lc_path(row: pd.Series, payload: dict[str, object] | None = None) -> Path | None:
    payload = payload or row_payload(row)
    candidates = [
        row.get('lc_path'),
        payload.get('lc_path'),
        payload.get('path'),
        payload.get('dat_path'),
        payload.get('asas_sn_id'),
        row.get('candidate_id'),
    ]
    for value in candidates:
        if value is None:
            continue
        resolved = resolve_local_lightcurve_path(value, run_dir=RUN_DIR, repo_root=REPO_ROOT)
        if resolved is not None and resolved.exists():
            return resolved
    return None


def mode_status(fits: pd.DataFrame, candidate_id: str, mode: str) -> str | None:
    if fits.empty:
        return None
    sub = fits[(fits['candidate_id'].astype(str) == str(candidate_id)) & (fits['mode'].astype(str) == str(mode))]
    if sub.empty:
        return None
    if 'updated_at' in sub.columns:
        sub = sub.sort_values('updated_at')
    return str(sub.iloc[-1].get('status') or '') or None


def planned_mode_action(fits: pd.DataFrame, candidate_id: str, mode: str) -> str:
    status = mode_status(fits, candidate_id, mode)
    if RERUN_EXISTING:
        return 'rerun_existing'
    if status in GOOD_STATUSES:
        return 'skipped_existing_good'
    if status:
        return f'retry_existing_{status}'
    return 'run_missing'


def active_controls_for_candidate(
    conn: sqlite3.Connection,
    candidate_id: str,
    payload: dict[str, object],
    *,
    lc_path: Path | None,
) -> dict[str, object]:
    defaults = control_defaults_for_candidate(
        conn,
        candidate_id,
        payload,
        lc_path=lc_path,
        plot_dir=PLOT_DIR,
        run_params=RUN_PARAMS,
        recompute=False,
    )
    controls = {key: defaults.get(key) for key in defaults.keys()}
    controls['_dustycult_window_source'] = str(defaults.get('source') or 'defaults')
    return controls


def select_fit_candidates(max_candidates: int | None = None) -> pd.DataFrame:
    data = reviewed_dippers.copy()
    if max_candidates is not None:
        data = data.head(int(max_candidates)).copy()
    return data.reset_index(drop=True)


def reload_existing_fits(candidate_ids: list[str] | None = None) -> pd.DataFrame:
    ids = [str(item) for item in (candidate_ids or reviewed_dippers['candidate_id'].astype(str).tolist())]
    if not ids:
        return pd.DataFrame()
    placeholders = ','.join(['?'] * len(ids))
    with db_connect(DB_PATH) as conn:
        return pd.read_sql_query(
            f'''
            SELECT candidate_id, mode, status, updated_at, runtime_sec, t0_jd, start_jd, end_jd,
                   n_input_points, n_curve_points, artifact_dir, error
            FROM dustycult_fits
            WHERE candidate_id IN ({placeholders})
            ORDER BY candidate_id, mode
            ''',
            conn,
            params=ids,
        )


def build_dry_run_queue(max_candidates: int | None = None) -> pd.DataFrame:
    fit_candidates = select_fit_candidates(max_candidates)
    current_fits = reload_existing_fits(fit_candidates['candidate_id'].astype(str).tolist())
    rows = []
    with db_connect(DB_PATH) as conn:
        for _, row in fit_candidates.iterrows():
            candidate_id = str(row['candidate_id'])
            payload = row_payload(row)
            lc_path = resolve_candidate_lc_path(row, payload)
            controls = active_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path)
            quick_status = mode_status(current_fits, candidate_id, 'quick')
            rows.append(
                {
                    'candidate_id': candidate_id,
                    'lc_found': lc_path is not None,
                    'lc_path': str(lc_path) if lc_path else '',
                    'quick_status': quick_status or 'missing',
                    'quick_action': planned_mode_action(current_fits, candidate_id, 'quick'),
                    'window_source': controls.get('_dustycult_window_source'),
                    'start_jd': controls.get('start_jd'),
                    't0_jd': controls.get('t0_jd'),
                    'end_jd': controls.get('end_jd'),
                    'dipper_score': row.get('dipper_score'),
                    'dipper_n_valid_dips': row.get('dipper_n_valid_dips'),
                    'dip_run_count': row.get('dip_run_count'),
                    'dip_best_morph': row.get('dip_best_morph'),
                }
            )
    return pd.DataFrame(rows)


## Dry-Run Queue

This cell does not write anything. Use it before the smoke test and again before the full run.

In [5]:
dry_run_queue = build_dry_run_queue(max_candidates=None)
display(dry_run_queue)
display(dry_run_queue.groupby(['quick_action', 'lc_found'], dropna=False).size().reset_index(name='n'))


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


,candidate_id,lc_found,lc_path,quick_status,quick_action,window_source,start_jd,t0_jd,end_jd,dipper_score,dipper_n_valid_dips,dip_run_count,dip_best_morph
0,stv_360777826205,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,recomputed_dip_run,2.458413e+06,2.458414e+06,2.458415e+06,21.400308,67.0,4.0,noise
1,stv_197569226514,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,recomputed_dip_run,2.459297e+06,2.459224e+06,2.459346e+06,-1.597394,11.0,6.0,noise
2,stv_395138016907,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,stored_event_columns,2.458081e+06,2.458201e+06,2.458321e+06,20.315220,58.0,15.0,gaussian
3,stv_163209415214,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,recomputed_dip_run,2.460378e+06,2.460375e+06,2.460398e+06,18.589905,20.0,6.0,skew_gaussian
4,stv_403727411589,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,stored_event_columns,2.458084e+06,2.458204e+06,2.458324e+06,18.884118,37.0,4.0,gaussian
5,stv_25770316308,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,recompute_deepest_point_fallback,2.458778e+06,2.458785e+06,2.458792e+06,20.531731,23.0,1.0,gaussian
6,stv_584116028406,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,stored_event_columns,2.460884e+06,2.460936e+06,2.460988e+06,-0.523791,9.0,4.0,gaussian
7,stv_154620038181,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,stored_event_columns,2.459738e+06,2.459812e+06,2.459885e+06,18.878795,28.0,12.0,gaussian
8,stv_180388640882,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,stored_event_columns,2.460384e+06,2.460504e+06,2.460624e+06,21.756072,40.0,3.0,gaussian
9,stv_17181027305,True,/Users/calder/code/malca/output/runs/dat3-full...,missing,run_missing,recompute_deepest_point_fallback,2.460317e+06,2.460324e+06,2.460331e+06,-3.302735,4.0,1.0,gaussian


,quick_action,lc_found,n
0,run_missing,True,69
1,skipped_existing_good,True,2


## Run DustyCult Fits

These cells write to the review DB and create/recreate DustyCult artifacts for candidates that are not skipped.

In [6]:
def record_result(results: list[dict[str, object]], candidate_id: str, mode: str, action: str, row: dict[str, object] | None = None, error: str = '') -> None:
    row = dict(row or {})
    results.append(
        {
            'candidate_id': str(candidate_id),
            'mode': mode,
            'action': action,
            'status': row.get('status', ''),
            'runtime_sec': row.get('runtime_sec'),
            't0_jd': row.get('t0_jd'),
            'start_jd': row.get('start_jd'),
            'end_jd': row.get('end_jd'),
            'n_input_points': row.get('n_input_points'),
            'n_curve_points': row.get('n_curve_points'),
            'artifact_dir': row.get('artifact_dir', ''),
            'error': row.get('error', error),
        }
    )


def run_fit_queue(max_candidates: int | None = None) -> pd.DataFrame:
    fit_candidates = select_fit_candidates(max_candidates)
    results: list[dict[str, object]] = []
    started = time.monotonic()

    with db_connect(DB_PATH) as conn:
        for idx, row in fit_candidates.iterrows():
            candidate_id = str(row['candidate_id'])
            payload = row_payload(row)
            lc_path = resolve_candidate_lc_path(row, payload)
            if lc_path is not None:
                payload['lc_path'] = str(lc_path)

            print(f'[{idx + 1}/{len(fit_candidates)}] {candidate_id}')
            fits_before = load_dustycult_fits(conn, candidate_id)
            controls = active_controls_for_candidate(conn, candidate_id, payload, lc_path=lc_path)

            quick_action = planned_mode_action(fits_before, candidate_id, 'quick')
            if quick_action == 'skipped_existing_good':
                status = mode_status(fits_before, candidate_id, 'quick') or 'skipped'
                record_result(results, candidate_id, 'quick', quick_action, {'status': status, **controls})
                print(f'  quick: skipped existing {status}')
                continue

            quick_row = run_dustycult_fit(
                conn,
                candidate_id,
                payload,
                db_path=DB_PATH,
                controls=controls,
                mode='quick',
                lc_path=lc_path,
                plot_dir=PLOT_DIR,
                run_params=RUN_PARAMS,
                project_path=DUSTYCULT_PROJECT,
                julia=JULIA,
            )
            record_result(results, candidate_id, 'quick', quick_action, quick_row)
            print(f"  quick: {quick_row.get('status')} {quick_action} {quick_row.get('error') or ''}")

    out = pd.DataFrame(results)
    elapsed = time.monotonic() - started
    print(f'Finished {len(out)} candidate-mode rows in {elapsed:.1f} s')
    return out


### Smoke Test

Run one candidate first. If it completes or records a sensible warning/failure, run the full queue cell below.

In [7]:
MAX_CANDIDATES = 1
smoke_results = run_fit_queue(max_candidates=MAX_CANDIDATES)
display(smoke_results)

if not smoke_results.empty:
    smoke_id = str(smoke_results.iloc[0]['candidate_id'])
    with db_connect(DB_PATH) as conn:
        smoke_fits = load_dustycult_fits(conn, smoke_id)
    display(smoke_fits)
    artifact_dir = smoke_results.iloc[0].get('artifact_dir')
    if artifact_dir:
        artifact_path = Path(str(artifact_dir))
        print(f'Artifact dir exists: {artifact_path.exists()} -> {artifact_path}')


[1/1] stv_360777826205
  quick: failed run_missing DustyCult needs at least 12 valid g/V points; found 3.
Finished 1 candidate-mode rows in 0.1 s


,candidate_id,mode,action,status,runtime_sec,t0_jd,start_jd,end_jd,n_input_points,n_curve_points,artifact_dir,error
0,stv_360777826205,quick,run_missing,failed,0.031034,2.458414e+06,2.458413e+06,2.458415e+06,3,0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...


,candidate_id,mode,status,created_at,updated_at,runtime_sec,artifact_dir,input_path,config_path,manifest_path,command_json,config_json,controls_json,window_json,stellar_json,posterior_json,summary_json,stderr_tail,stdout_tail,error,t0_jd,start_jd,end_jd,n_input_points,n_curve_points
0,stv_360777826205,quick,failed,2026-07-12T22:37:29.056526+00:00,2026-07-12T22:37:29.056526+00:00,0.031034,/Users/calder/code/malca/output/runs/dat3-full...,None,None,None,[],{},"{""alpha_center"":0.0,""alpha_width"":2.0,""b_cente...","{""end_jd"":2458415.08134,""lc_path"":""/Users/cald...",None,None,"{""quality"":{""after_t0_points"":1,""band_counts"":...",,,DustyCult needs at least 12 valid g/V points; ...,2.458414e+06,2.458413e+06,2.458415e+06,3,0


Artifact dir exists: False -> /Users/calder/code/malca/output/runs/dat3-full-extended_2026-07-01-v4/review/dustycult/stv_360777826205/quick


### Full Remaining Quick Run

This sets `MAX_CANDIDATES = None`. Existing `ok` or `warning` quick fits are skipped, including the smoke-test result if it succeeded or warned.

In [8]:
MAX_CANDIDATES = None
full_results = run_fit_queue(max_candidates=MAX_CANDIDATES)
display(full_results)
display(full_results.groupby(['action', 'status'], dropna=False).size().reset_index(name='n'))


[1/71] stv_360777826205
  quick: failed retry_existing_failed DustyCult needs at least 12 valid g/V points; found 3.
[2/71] stv_197569226514
  quick: failed run_missing DustyCult t0 is outside the selected fit window. DustyCult needs at least 12 valid g/V points; found 7.
[3/71] stv_395138016907
  quick: warning run_missing DustyCult reported 18 divergent transitions.
[4/71] stv_163209415214
  quick: failed run_missing DustyCult t0 is outside the selected fit window.
[5/71] stv_403727411589
  quick: warning run_missing DustyCult reported 111 divergent transitions.
[6/71] stv_25770316308
  quick: warning run_missing DustyCult has only 12 valid g/V points. DustyCult input contains only one ASAS-SN band. DustyCult input spans only 11.99 days. DustyCult input has sparse coverage on one side of t0. DustyCult reported 8 divergent transitions.
[7/71] stv_584116028406
  quick: warning run_missing DustyCult input contains only one ASAS-SN band. DustyCult reported 3 divergent transitions.
[8/71]

,candidate_id,mode,action,status,runtime_sec,t0_jd,start_jd,end_jd,n_input_points,n_curve_points,artifact_dir,error
0,stv_360777826205,quick,retry_existing_failed,failed,0.003039,2.458414e+06,2.458413e+06,2.458415e+06,3.0,0.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...
1,stv_197569226514,quick,run_missing,failed,0.018269,2.459224e+06,2.459297e+06,2.459346e+06,7.0,0.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult t0 is outside the selected fit windo...
2,stv_395138016907,quick,run_missing,warning,3406.404095,2.458201e+06,2.458081e+06,2.458321e+06,54.0,54.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult reported 18 divergent transitions.
3,stv_163209415214,quick,run_missing,failed,0.002573,2.460375e+06,2.460378e+06,2.460398e+06,14.0,0.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult t0 is outside the selected fit window.
4,stv_403727411589,quick,run_missing,warning,622.792783,2.458204e+06,2.458084e+06,2.458324e+06,89.0,89.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult reported 111 divergent transitions.
5,stv_25770316308,quick,run_missing,warning,399.436591,2.458785e+06,2.458778e+06,2.458792e+06,12.0,12.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult has only 12 valid g/V points. DustyC...
6,stv_584116028406,quick,run_missing,warning,4010.612331,2.460936e+06,2.460884e+06,2.460988e+06,80.0,80.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
7,stv_154620038181,quick,run_missing,warning,3193.433684,2.459812e+06,2.459738e+06,2.459885e+06,89.0,89.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
8,stv_180388640882,quick,run_missing,warning,6886.685287,2.460504e+06,2.460384e+06,2.460624e+06,152.0,152.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
9,stv_17181027305,quick,run_missing,failed,0.003559,2.460324e+06,2.460317e+06,2.460331e+06,3.0,0.0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...


,action,status,n
0,retry_existing_failed,failed,1
1,run_missing,failed,14
2,run_missing,ok,3
3,run_missing,warning,51
4,skipped_existing_good,ok,2


## Summary And Example Display

Use this after the smoke test or full run. It reloads from SQLite instead of trusting in-memory results.

In [9]:
candidate_ids = reviewed_dippers['candidate_id'].astype(str).tolist()
final_fits = reload_existing_fits(candidate_ids)

display(final_fits.groupby(['mode', 'status'], dropna=False).size().reset_index(name='n') if not final_fits.empty else Markdown('No DustyCult fits found.'))
display(final_fits.sort_values(['candidate_id', 'mode']) if not final_fits.empty else final_fits)

quick_final = final_fits[final_fits['mode'].astype(str).eq('quick')].copy() if not final_fits.empty else pd.DataFrame()
quick_good = quick_final[quick_final['status'].astype(str).isin(GOOD_STATUSES)].copy() if not quick_final.empty else pd.DataFrame()
print(f'Quick ok/warning fits: {len(quick_good)} / {len(reviewed_dippers)}')


def _display_rows_table(rows: list[tuple[str, str]] | list[tuple[str, str, str, str]]) -> None:
    if not rows:
        return
    frame = pd.DataFrame(rows)
    display(frame)


def _load_selected_dustycult_fit(candidate_id: str, mode: str | None = None):
    with db_connect(DB_PATH) as conn:
        fits = load_dustycult_fits(conn, candidate_id)
        fit_row = select_dustycult_display_row(fits, mode=mode)
        if fit_row is None:
            raise ValueError(f'No DustyCult fit row found for {candidate_id}')
        selected_mode = str(fit_row.get('mode') or 'quick')
        curves = load_dustycult_curve(conn, candidate_id, selected_mode)
    return fit_row, curves


def display_dustycult_review_panel(candidate_id: str, mode: str | None = None) -> None:
    fit_row, curves = _load_selected_dustycult_fit(candidate_id, mode)
    selected_mode = str(fit_row.get('mode') or 'quick')
    display(Markdown(f'### `{candidate_id}` DustyCult `{selected_mode}`'))
    _display_rows_table(dustycult_fit_metadata_rows(fit_row))
    if curves is not None and not curves.empty:
        display(build_dustycult_fit_figure(curves, fit_row, theme='white'))
    try:
        display(build_dustycult_occulter_figure(fit_row, theme='white', grid_n=501))
    except Exception as exc:
        display(Markdown(f'Occulter plot unavailable: `{exc}`'))
    _display_rows_table(dustycult_geometry_rows(fit_row))
    _display_rows_table(dustycult_posterior_rows(fit_row, limit=None))


if quick_good.empty:
    display(Markdown('No ok/warning quick fits are available yet.'))
else:
    example = quick_good.sort_values(['status', 'candidate_id']).iloc[0]
    display_dustycult_review_panel(str(example['candidate_id']), EXAMPLE_PLOT_MODE)


,mode,status,n
0,quick,failed,15
1,quick,ok,4
2,quick,warning,52


,candidate_id,mode,status,updated_at,runtime_sec,t0_jd,start_jd,end_jd,n_input_points,n_curve_points,artifact_dir,error
0,stv_111669273145,quick,failed,2026-07-13T04:49:26.444364+00:00,0.028511,2.460276e+06,2.460269e+06,2.460280e+06,11,0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...
1,stv_111669291649,quick,warning,2026-07-14T19:08:35.117218+00:00,1531.123928,2.457349e+06,2.457252e+06,2.457447e+06,75,75,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
2,stv_111669557747,quick,warning,2026-07-15T00:55:50.537948+00:00,253.809339,2.458325e+06,2.458297e+06,2.458353e+06,19,19,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult has only 19 valid g/V points. DustyC...
3,stv_128850429575,quick,warning,2026-07-14T15:44:44.952563+00:00,932.439090,2.457504e+06,2.457384e+06,2.457624e+06,35,35,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
4,stv_146029419304,quick,failed,2026-07-15T00:02:35.107381+00:00,0.002144,2.458320e+06,2.458313e+06,2.458327e+06,5,0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...
5,stv_154620038181,quick,warning,2026-07-13T01:51:22.140575+00:00,3193.433684,2.459812e+06,2.459738e+06,2.459885e+06,89,89,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...
6,stv_163209415214,quick,failed,2026-07-12T23:34:15.731540+00:00,0.002573,2.460375e+06,2.460378e+06,2.460398e+06,14,0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult t0 is outside the selected fit window.
7,stv_163209590869,quick,warning,2026-07-14T16:46:57.243325+00:00,1218.128649,2.459435e+06,2.459387e+06,2.459482e+06,60,60,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band.
8,stv_17181027305,quick,failed,2026-07-13T03:46:08.973586+00:00,0.003559,2.460324e+06,2.460317e+06,2.460331e+06,3,0,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult needs at least 12 valid g/V points; ...
9,stv_180388640882,quick,warning,2026-07-13T03:46:08.883158+00:00,6886.685287,2.460504e+06,2.460384e+06,2.460624e+06,152,152,/Users/calder/code/malca/output/runs/dat3-full...,DustyCult input contains only one ASAS-SN band...


Quick ok/warning fits: 56 / 71


### `stv_240519504803` DustyCult `quick`

,0,1
0,mode,quick
1,status,ok
2,runtime,2201.091 s
3,window,2.5e+06 to 2.5e+06
4,bands,"V:15, g:16"
5,input span,87.01 d
6,baseline,gp_masked
7,artifact,/Users/calder/code/malca/output/runs/dat3-full...


,0,1
0,t0,2.4583e+06
1,v,0.55279
2,b,0.03883
3,b / R_star,0.00647
4,tau0,0.45805
5,lambda0 [nm],656.89032
6,alpha,13.76961
7,sigma_y,0.94619
8,sigma_x_plus,11.29524
9,sigma_x_minus,11.62543


,0,1,2,3
0,alpha,13.7696,12.536,14.7664
1,b,0.0388,-0.5743,0.5537
2,lambda0,656.8903,599.0587,723.2834
3,sigma_x_minus,11.6254,9.4853,14.3704
4,sigma_x_plus,11.2952,9.391,14.3577
5,sigma_y,0.9462,0.9153,0.9875
6,t0,2.458e+06,2.458e+06,2.458e+06
7,tau0,0.4581,0.1098,1.6489
8,v,0.5528,0.4694,0.6864
